In [ ]:
%%time

!pip install -q pandas
!pip install -q pyarrow
!pip install -q transformers[torch]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import torch

In [ ]:
MODEL_DIR_NAME = 'merged-model'
TEST_DATA_NAME = 'test-data.parquet.gzip'
CHECKPOINT = MODEL_DIR_NAME
SEQUENCE_LENGTH = 512

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, torch_dtype="auto"
).to(device)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

In [ ]:
def infer(texts: list[str], batch_size: int = 512):
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=SEQUENCE_LENGTH,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            confidents = torch.softmax(outputs.logits, dim=-1)
            results.extend(score.item() for score in confidents[:, 1])
        # 主動釋放記憶體
        del inputs, outputs, confidents
        torch.cuda.empty_cache()
    return results

In [ ]:
%%time

data = pd.read_parquet(TEST_DATA_NAME)

In [ ]:
texts = data['text'].to_list()

In [ ]:
%%time

data['score'] = infer(texts)

In [ ]:
grouped = data.groupby("article_id").agg({"score":list, "text": ''.join,"is_spam":sum})

In [ ]:
grouped["max"] = grouped.apply(lambda row: max(row["score"]) ,axis=1)

In [ ]:
step = 0.01
s = 0.8
while s < 1:
    print(s)
    fp = len(grouped.query("is_spam == 0").query(f"max >= {s}"))
    fn = len(grouped.query("is_spam > 0").query(f"max < {s}"))
    print(f"total:{fp+fn} bias:{2*fp + fn} fp:{fp} fn:{fn}")
    s += step

In [ ]:
grouped.to_parquet('scored-' + TEST_DATA_NAME, compression='gzip') 